In [1]:
# Import the libraries required for text processing, topic modeling,
# supervised classification, and metric analysis
import pandas as pd
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier

In [2]:
# Download NLTK resources for tokenization, lemmatization, and grammatical tagging
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')


True

In [3]:
# Function to remove emojis from text using a regular expression that filters the corresponding Unicode characters
def remove_emojis(text):
    if isinstance(text, str):
        emoji_pattern = re.compile(pattern="[" 
            "\U0001F600-\U0001F64F" "\U0001F300-\U0001F5FF"
            "\U0001F680-\U0001F6FF" "\U0001F700-\U0001F77F"
            "\U0001F780-\U0001F7FF" "\U0001F800-\U0001F8FF"
            "\U0001F900-\U0001F9FF" "\U0001FA00-\U0001FA6F"
            "\U0001FA70-\U0001FAFF" "\U00002702-\U000027B0"
            "\U000024C2-\U0001F251" "]+", flags=re.UNICODE)
        return emoji_pattern.sub(r'', text)
    return text

# Initialize the WordNet-based lemmatizer
lemmatizer = WordNetLemmatizer()

# Auxiliary function that obtains the grammatical tag (part-of-speech) of each word to improve lemmatization
def get_pos_tag(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ, "N": wordnet.NOUN, "V": wordnet.VERB, "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

# Function that applies lemmatization to the text:
# Tokenizes the text, applies POS-based lemmatization, and returns the reconstructed text
def lemmatize_text(text):
    words = nltk.word_tokenize(text)
    lemmatized_words = [lemmatizer.lemmatize(w, get_pos_tag(w)) for w in words]
    return ' '.join(lemmatized_words)


In [4]:
# Load the SMS dataset from an online source
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

# Convert class labels: 'ham' is encoded as 0 and 'spam' as 1
df['label'] = df['label'].replace({'ham': 0, 'spam': 1})

# Apply text cleaning and normalization:
# remove emojis and then lemmatize
df['message'] = df['message'].apply(remove_emojis).apply(lemmatize_text)

# Define a list of stopwords to be excluded from the feature model
stopwords = ['a', 'an', 'the', 'in', 'on', 'at', 'to', 'of', 'and', 'or',
             'is', 'it', 'for', 'with', 'that', 'this', 'as', 'was', 'be',
             'are', 'were', 'been', 'from', 'by', 'about', 'into', 'out',
             'up', 'down', 'over', 'under', 'then', 'than', 'so', 'but', 'not']

# Split the data into independent variables (X_text) and dependent variables (y)
X_text = df["message"]
y = df["label"]


In [5]:
# Define the values of k for the number of topics in the LDA model
k_values = [10]
results = []

In [6]:
i=0
# Additional import (not necessary) of another boosting model
from sklearn.ensemble import GradientBoostingClassifier

# Configure the stratified cross-validation method (5 splits)
# This ensures a balanced class distribution in each split
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Main training and validation loop for each value of k
for k in k_values:
    aucs = []  # List to store the AUC scores for each split
    for train_index, test_index in skf.split(X_text, y):
        # Split the data into training and test sets
        X_train_raw, X_test_raw = X_text.iloc[train_index], X_text.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        # Vectorize the texts using a bag-of-words representation with a limit of 10,000 features
        vectorizer = CountVectorizer(max_features=10000, stop_words=stopwords)
        X_train_counts = vectorizer.fit_transform(X_train_raw)
        X_test_counts = vectorizer.transform(X_test_raw)

        # Apply the LDA model for dimensionality reduction and topic extraction
        lda = LatentDirichletAllocation(n_components=k, random_state=42)
        X_train_topics = lda.fit_transform(X_train_counts)
        X_test_topics = lda.transform(X_test_counts)

        # Initialize and train the XGBoost classification model
        model = XGBClassifier(
            max_depth=6,            # Maximum depth of the decision trees
            n_estimators=500,       # Total number of trees
            learning_rate=0.01,     # Learning rate
            random_state=42         # Seed for reproducibility
        )
        model.fit(X_train_topics, y_train)
        print("Iteration: "+str(i) )  # Print progress information for the current iteration
        i=i+1
        # Predict probabilities for the positive class and compute the AUC-ROC metric
        probs = model.predict_proba(X_test_topics)[:, 1]
        auc = roc_auc_score(y_test, probs)
        aucs.append(auc)
        print("AUC-ROC: Fold " + str(i) +  " : " + str(auc))  # Print the AUC score for the current fold

    # Store the mean AUC score for this value of k
    results.append({"k": k, "mean_auc_roc": np.mean(aucs)})

Iteration: 0
AUC-ROC: Fold 1 : 0.9745492227979276
Iteration: 1
AUC-ROC: Fold 2 : 0.9694024179620035
Iteration: 2
AUC-ROC: Fold 3 : 0.965698786382446
Iteration: 3
AUC-ROC: Fold 4 : 0.9655492575720694
Iteration: 4
AUC-ROC: Fold 5 : 0.9533261466773307


In [7]:
# Organize the results
df_results = pd.DataFrame(results)

# Display the results table
print("\n Cross-validation results (mean AUC-ROC for k=10):\n")
print(df_results.to_string(index=False, float_format="%.4f"))


 Cross-validation results (mean AUC-ROC for k=10):

 k  mean_auc_roc
10        0.9657
